### Cốt lõi của thuật toán (DLS và IDS)

In [1]:
from itertools import count

CUTOFF = "cutoff"

class Node:
    def __init__(self, state, parent=None, action=None, depth=0):
        self.state = state
        self.parent = parent
        self.action = action
        self.depth = depth

    def get_path(self):
        path = []
        cur_node = self
        while cur_node:
            path.append((cur_node.state, cur_node.action))
            cur_node = cur_node.parent
        return path[::-1]

class Problem:
    def __init__(self, start_state, target_state, get_moves, apply_move, step_cost=None):
        self.start_state = start_state
        self.target_state = target_state
        self.get_moves = get_moves
        self.apply_move = apply_move
        self.step_cost = step_cost

def is_goal(problem, cur_state):
    if callable(problem.target_state):
        return problem.target_state(cur_state)
    return cur_state == problem.target_state

def child_node(problem, parent, action):
    next_state = problem.apply_move(parent.state, action)
    return Node(next_state, parent, action, parent.depth + 1)

def describe(result):
    if result is CUTOFF:
        return "CUTOFF"
    if result is None:
        return "failure"
    return f"tìm thấy lời giải {len(result) - 1} bước"

def report(result, print_state):
    if not isinstance(result, list):
        print(describe(result))
        return

    print(f"SOLUTION ({len(result) - 1} bước)")
    for step_idx in range(len(result)):
        state, action = result[step_idx]
        if step_idx == len(result) - 1:
            print(f"Bước {step_idx} ({action}) (GOAL):")
        else:
            print(f"Bước {step_idx} ({action}):")
        print_state(state)

def run_counted(search, problem, *args, **kwargs):
    generated = 0

    def counted_apply(state, action):
        nonlocal generated
        generated += 1
        return problem.apply_move(state, action)

    counted = Problem(problem.start_state, problem.target_state, problem.get_moves, counted_apply, problem.step_cost)
    result = search(counted, *args, **kwargs)
    return result, generated

### Bản 1: DLS và IDS đệ quy theo mã giả

In [2]:
def recursive_dls(node, problem, limit):
    if is_goal(problem, node.state):
        return node
    if limit == 0:
        return CUTOFF

    cutoff_occurred = False
    for action in problem.get_moves(node.state):
        child = child_node(problem, node, action)

        result = recursive_dls(child, problem, limit - 1)
        if result is CUTOFF:
            cutoff_occurred = True
        elif result is not None:
            return result

    return CUTOFF if cutoff_occurred else None

def DLS(problem: Problem, limit):
    result = recursive_dls(Node(problem.start_state), problem, limit)
    if isinstance(result, Node):
        return result.get_path()
    return result

def IDS(problem: Problem, max_depth=None):
    for depth in count():
        if max_depth is not None and depth > max_depth:
            return CUTOFF

        result = DLS(problem, depth)
        if result is not CUTOFF:
            return result

### Bản 2: DLS và IDS đệ quy tối ưu

In [3]:
def is_cycle(node):
    ancestor = node.parent
    while ancestor is not None:
        if ancestor.state == node.state:
            return True
        ancestor = ancestor.parent
    return False

def recursive_dls_opt(node, problem, limit):
    if is_goal(problem, node.state):
        return node
    if limit == 0:
        return CUTOFF

    cutoff_occurred = False
    for action in problem.get_moves(node.state):
        child = child_node(problem, node, action)
        if is_cycle(child):
            continue

        result = recursive_dls_opt(child, problem, limit - 1)
        if result is CUTOFF:
            cutoff_occurred = True
        elif result is not None:
            return result

    return CUTOFF if cutoff_occurred else None

def DLS_opt(problem: Problem, limit):
    result = recursive_dls_opt(Node(problem.start_state), problem, limit)
    if isinstance(result, Node):
        return result.get_path()
    return result

def IDS_opt(problem: Problem, max_depth=None):
    for depth in count():
        if max_depth is not None and depth > max_depth:
            return CUTOFF

        result = DLS_opt(problem, depth)
        if result is not CUTOFF:
            return result

### Bài toán 8-Puzzle

In [4]:
def find_blank(cur_board):
    for row_idx in range(3):
        for col_idx in range(3):
            if cur_board[row_idx][col_idx] == 0:
                return row_idx, col_idx

def actions_8puzzle(cur_board):
    blank_r, blank_c = find_blank(cur_board)
    valid_moves = []

    if blank_r > 0: valid_moves.append("Lên")
    if blank_r < 2: valid_moves.append("Xuống")
    if blank_c > 0: valid_moves.append("Trái")
    if blank_c < 2: valid_moves.append("Phải")

    return valid_moves

def result_8puzzle(cur_board, move):
    blank_r, blank_c = find_blank(cur_board)

    new_r, new_c = blank_r, blank_c
    if move == "Lên":
        new_r -= 1
    elif move == "Xuống":
        new_r += 1
    elif move == "Trái":
        new_c -= 1
    elif move == "Phải":
        new_c += 1

    grid = [list(cur_board[0]), list(cur_board[1]), list(cur_board[2])]
    grid[blank_r][blank_c] = grid[new_r][new_c]
    grid[new_r][new_c] = 0

    return (tuple(grid[0]), tuple(grid[1]), tuple(grid[2]))

def print_state(board):
    for r in range(3):
        print(f"[{board[r][0]} {board[r][1]} {board[r][2]}]")
    print()

if __name__ == "__main__":
    target_board = ((1, 2, 3), (4, 5, 6), (7, 8, 0))

    start_board = ((1, 2, 3), (4, 0, 6), (7, 5, 8))
    puzzle = Problem(start_board, target_board, actions_8puzzle, result_8puzzle)

    print("=== DLS với từng giới hạn ===")
    for limit in range(4):
        print(f"limit = {limit}: bản 1 -> {describe(DLS(puzzle, limit))} | bản 2 -> {describe(DLS_opt(puzzle, limit))}")

    print()
    print("=== IDS bản 1 (theo mã giả của cô) ===")
    report(IDS(puzzle), print_state)

    print("=== IDS bản 2 (tối ưu) ===")
    report(IDS_opt(puzzle), print_state)

    print("=== So sánh số nút sinh ra khi chạy IDS ===")
    print(f"{'Độ sâu lời giải':>16} | {'Bản 1':>10} | {'Bản 2':>10} | {'Bản 1 / Bản 2':>13}")
    scramble_moves = ["Lên", "Trái", "Lên", "Trái", "Xuống", "Phải", "Xuống", "Phải"]
    for steps in (2, 4, 6, 8):
        board = target_board
        for move in scramble_moves[:steps]:
            board = result_8puzzle(board, move)
        task = Problem(board, target_board, actions_8puzzle, result_8puzzle)
        path1, generated1 = run_counted(IDS, task)
        path2, generated2 = run_counted(IDS_opt, task)
        print(f"{len(path2) - 1:>16} | {generated1:>10} | {generated2:>10} | {generated1 / generated2:>12.1f}x")

=== DLS với từng giới hạn ===
limit = 0: bản 1 -> CUTOFF | bản 2 -> CUTOFF
limit = 1: bản 1 -> CUTOFF | bản 2 -> CUTOFF
limit = 2: bản 1 -> tìm thấy lời giải 2 bước | bản 2 -> tìm thấy lời giải 2 bước
limit = 3: bản 1 -> tìm thấy lời giải 2 bước | bản 2 -> tìm thấy lời giải 2 bước

=== IDS bản 1 (theo mã giả của cô) ===
SOLUTION (2 bước)
Bước 0 (None):
[1 2 3]
[4 0 6]
[7 5 8]

Bước 1 (Xuống):
[1 2 3]
[4 5 6]
[7 0 8]

Bước 2 (Phải) (GOAL):
[1 2 3]
[4 5 6]
[7 8 0]

=== IDS bản 2 (tối ưu) ===
SOLUTION (2 bước)
Bước 0 (None):
[1 2 3]
[4 0 6]
[7 5 8]

Bước 1 (Xuống):
[1 2 3]
[4 5 6]
[7 0 8]

Bước 2 (Phải) (GOAL):
[1 2 3]
[4 5 6]
[7 8 0]

=== So sánh số nút sinh ra khi chạy IDS ===
 Độ sâu lời giải |      Bản 1 |      Bản 2 | Bản 1 / Bản 2
               2 |         19 |         19 |          1.0x
               4 |         87 |         66 |          1.3x
               6 |       1251 |        329 |          3.8x
               8 |       5479 |        838 |          6.5x


### Bài toán Vacuum Robot

In [5]:
def actions_vacuum(cur_state):
    return ["Hút", "Trái", "Phải"]

def result_vacuum(cur_state, move):
    pos, room_a, room_b = cur_state

    if move == "Trái":
        return ("A", room_a, room_b)
    elif move == "Phải":
        return ("B", room_a, room_b)
    elif move == "Hút":
        if pos == "A":
            return (pos, "Sạch", room_b)
        else:
            return (pos, room_a, "Sạch")

def goal_vacuum(cur_state):
    return cur_state[1] == "Sạch" and cur_state[2] == "Sạch"

def print_vacuum_state(cur_state):
    pos, room_a, room_b = cur_state
    print(f"Robot ở phòng {pos} | Phòng A: {room_a} | Phòng B: {room_b}")

if __name__ == "__main__":
    start_vacuum = ("A", "Bẩn", "Bẩn")
    vacuum_puzzle = Problem(start_vacuum, goal_vacuum, actions_vacuum, result_vacuum)

    print("=== DLS với từng giới hạn ===")
    for limit in range(5):
        print(f"limit = {limit}: bản 1 -> {describe(DLS(vacuum_puzzle, limit))} | bản 2 -> {describe(DLS_opt(vacuum_puzzle, limit))}")

    print()
    print("=== IDS bản 1 (theo mã giả của cô) ===")
    report(IDS(vacuum_puzzle), print_vacuum_state)

    print("=== IDS bản 2 (tối ưu) ===")
    report(IDS_opt(vacuum_puzzle), print_vacuum_state)

    print("=== Số nút sinh ra khi chạy IDS ===")
    _, generated1 = run_counted(IDS, vacuum_puzzle)
    _, generated2 = run_counted(IDS_opt, vacuum_puzzle)
    print(f"Bản 1: sinh {generated1} nút")
    print(f"Bản 2: sinh {generated2} nút")

    print()
    print("=== Đích không thể đạt tới ===")
    impossible = Problem(start_vacuum, ("C", "Sạch", "Sạch"), actions_vacuum, result_vacuum)
    print("Bản 1, DLS limit = 6  :", describe(DLS(impossible, 6)))
    print("Bản 1, IDS (max_depth=6):", describe(IDS(impossible, max_depth=6)), "(không bao giờ kết luận được failure)")
    print("Bản 2, DLS limit = 20 :", describe(DLS_opt(impossible, 20)))
    print("Bản 2, IDS            :", describe(IDS_opt(impossible)))

=== DLS với từng giới hạn ===
limit = 0: bản 1 -> CUTOFF | bản 2 -> CUTOFF
limit = 1: bản 1 -> CUTOFF | bản 2 -> CUTOFF
limit = 2: bản 1 -> CUTOFF | bản 2 -> CUTOFF
limit = 3: bản 1 -> tìm thấy lời giải 3 bước | bản 2 -> tìm thấy lời giải 3 bước
limit = 4: bản 1 -> tìm thấy lời giải 4 bước | bản 2 -> tìm thấy lời giải 3 bước

=== IDS bản 1 (theo mã giả của cô) ===
SOLUTION (3 bước)
Bước 0 (None):
Robot ở phòng A | Phòng A: Bẩn | Phòng B: Bẩn
Bước 1 (Hút):
Robot ở phòng A | Phòng A: Sạch | Phòng B: Bẩn
Bước 2 (Phải):
Robot ở phòng B | Phòng A: Sạch | Phòng B: Bẩn
Bước 3 (Hút) (GOAL):
Robot ở phòng B | Phòng A: Sạch | Phòng B: Sạch
=== IDS bản 2 (tối ưu) ===
SOLUTION (3 bước)
Bước 0 (None):
Robot ở phòng A | Phòng A: Bẩn | Phòng B: Bẩn
Bước 1 (Hút):
Robot ở phòng A | Phòng A: Sạch | Phòng B: Bẩn
Bước 2 (Phải):
Robot ở phòng B | Phòng A: Sạch | Phòng B: Bẩn
Bước 3 (Hút) (GOAL):
Robot ở phòng B | Phòng A: Sạch | Phòng B: Sạch
=== Số nút sinh ra khi chạy IDS ===
Bản 1: sinh 26 nút
Bản 2: si